In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
df = pd.read_csv("powerplant_data.csv")
# AT = temperature
# v = vaccum
# ap = pressure
# eh = humidity

In [3]:
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
X = df.drop("PE",axis = 1)
y = df["PE"]

In [6]:
#split out data
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.2,random_state = 42
)

In [7]:
df.shape

(9568, 5)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [9]:
#build a tensor 
X_train_tensor = torch.tensor(X_train_scaled,dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [10]:
from torch.utils.data import TensorDataset,DataLoader
#for tensor dataset 
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
testing_dataset = TensorDataset(X_test_tensor,y_test_tensor)

#for dataloader
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(testing_dataset,batch_size=32)

## Deep learning

In [11]:
#Define ANN Model
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model= nn.Sequential(
            #1st hidden layer
            nn.Linear(X_train.shape[1],6),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),


            #output layer
            nn.Linear(6,1),
        )


    #logic for forward propagation
    def forward(self,x):
        return self.model(x)

In [12]:
import torch.optim as optim
model = ANN()

#define loss and optimizer
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [16]:
#Train our ann model through multiple epochs

train_losses = []
validation_losses  = []
epochs=100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 #total training loss for 1 epoch

    for xb,yb in train_loader:
        #xb = features of 1 batch
        # yb = labels of 1 batch
        optimizer.zero_grad()

        #forward propagation
        outputs = model(xb) #predicted output for this batch

        #compute loss
        loss = crietrion(outputs,yb)

        #back propagation compute loss
        loss.backward()

        #update paramas
        optimizer.step()

        running_loss += loss.item() #loss is a tensor ==> py float
        
    epoch_train_loss = running_loss / len(train_loader)
    train_losses.append(epoch_train_loss)


    #Validation
    model.eval()
    running_val_loss = 0.0
    

    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs,yb)
            running_val_loss += loss.item() 

    epoch_val_loss = running_val_loss/len(test_loader)
    validation_losses.append(epoch_val_loss)

    print(f"epoch ${epoch+1}/{epochs} ==> training loss = ${epoch_train_loss} & validation loss = ${epoch_val_loss}")



    

epoch $1/100 ==> training loss = $152490.96005859374 & validation loss = $124263.80546875
epoch $2/100 ==> training loss = $94751.95830078125 & validation loss = $68100.37708333334
epoch $3/100 ==> training loss = $49490.76582845052 & validation loss = $35586.18450520833
epoch $4/100 ==> training loss = $27147.713024902343 & validation loss = $21145.863639322917
epoch $5/100 ==> training loss = $17347.140991210938 & validation loss = $14508.301236979167
epoch $6/100 ==> training loss = $12466.935186767578 & validation loss = $10710.969067382812
epoch $7/100 ==> training loss = $9256.448653157551 & validation loss = $7811.838387044271
epoch $8/100 ==> training loss = $6647.4860387166345 & validation loss = $5529.104512532552
epoch $9/100 ==> training loss = $4677.664111328125 & validation loss = $3839.541876220703
epoch $10/100 ==> training loss = $3186.742817179362 & validation loss = $2583.275956217448
epoch $11/100 ==> training loss = $2120.7309577941896 & validation loss = $1724.419

In [22]:
import matplotlib.pyplot as plt

loss_df = pd.DataFrame({
    "Training Loss":train_losses,
    "Validation Loss":validation_losses
})
plt.figure(figsize(8,6))
plt.plot(loss_df["Training Loss"],label = "Training Loss")
plt.plot(loss_df["Validation Loss"],label ="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Losses")
plt.legend()

NameError: name 'fig_size' is not defined